In [ ]:
import papermill as pm
import duckdb, subprocess, os, time
from joblib import Parallel, delayed

In [ ]:
species_list = [
    "Protonotaria citrea",
    "Limnothlypis swainsonii",
    "Setophaga americana",
    "Empidonax virescens",
    "Coccyzus americanus",
    "Vireo griseus",
    "Setophaga cerulea",
    "Hylocichla mustelina",
    "Parkesia motacilla",
    "Geothlypis formosa",
    "Archilochus colubris",
    "Elanoides forficatus",
    "Vireo flavifrons",
    "Buteo lineatus",
    "Setophaga dominica",
    "Setophaga citrina",
    "Dryocopus pileatus",
    "Meleagris gallopavo",
    "Sphyrapicus varius",
    "Odocoileus virginianus",     # white tailed deer
    "Ursus americanus",           # Black bear
    "Anaxyrus americanus",        # American Toad
    "Anaxyrus fowleri",           # Fowler's Toad
    "Gastrophryne carolinensis",  # Eastern Narrow-mouthed Toad
    "Hyla avivoca",               # Bird-voiced Treefrog
    "Hyla chrysoscelis",          # Cope's Gray Treefrog
    "Hyla cinerea",               # Green Treefrog
    "Hyla squirella",             # Squirrel Treefrog    
    "Hyla versicolor",            # Gray Treefrog
    "Lithobates catesbeianus",    # American Bullfrog
    "Lithobates clamitans",       # Bronze Frog
    "Lithobates palustris",       # Pickerel Frog
    "Lithobates sphenocephalus",  # Southern Leopard Frog
    "Pseudacris crucifer",        # Spring Peeper
    "Pseudacris fouquettei",      # Cajun Chorus Frog
    "Kinosternon subrubrum",      # Eastern Mud Turtle
    "Apalone spinifera",          # Spiny Softshell Turtle   
    "Macrochelys temmincki"       # Alligator Snapping Turtle    
    ]
paramdir = '/mnt/f/readyparams/param_csvs'
outputdir = '/mnt/f/readyparams/ppp_paramsoutput'
jobs = 10
os.makedirs(outputdir, exist_ok=True)

In [ ]:
species_list = [s.replace(" ", "_").lower() for s in species_list]
def f(x):
    try:
        spoutputdir = os.path.join(outputdir,x)
        os.makedirs(spoutputdir, exist_ok=True)
        print('run',x)
        if jobs >1:
            subprocess.run(pm.execute_notebook(f'maxent_model.ipynb',os.path.join(spoutputdir,'{0}.ipynb'.format(x)),parameters=dict(parambasedir=paramdir, baseoutputdir=spoutputdir, sp=x)), shell=True)
        else:
            pm.execute_notebook(f'maxent_model.ipynb',os.path.join(spoutputdir,'{0}.ipynb'.format(x)),parameters=dict(parambasedir=paramdir, baseoutputdir=spoutputdir, sp=x))
        return (x, 'run complete')
    except Exception as e:
        return (x,'fail', e)

In [ ]:
completed = Parallel(n_jobs=10, verbose=0)(delayed(f)(species_list[x]) for x in range(len(species_list)))

In [ ]:
print('## Failed species ##')
print('')
for i, result in enumerate(completed):
    if result[1] == 'fail':
        print(result[0])